In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.4 MB/s eta 0:00:00


In [2]:
!python /content/sample_data/ADV_STEP1_multiturn_sft.py

Loading ToolBench dataset ...
README.md: 3.10kB [00:00, 3.46MB/s]
data/train-00000-of-00004.parquet: 100% 135M/135M [00:04<00:00, 29.9MB/s]
data/train-00001-of-00004.parquet: 100% 134M/134M [00:04<00:00, 33.4MB/s]
data/train-00002-of-00004.parquet: 100% 135M/135M [00:04<00:00, 31.9MB/s]
data/train-00003-of-00004.parquet: 100% 134M/134M [00:04<00:00, 33.5MB/s]
data/validation-00000-of-00001.parquet: 100% 2.01M/2.01M [00:00<00:00, 4.88MB/s]
Generating train split: 100% 187542/187542 [00:19<00:00, 9556.12 examples/s] 
Generating validation split: 100% 762/762 [00:00<00:00, 27522.11 examples/s]
Total ToolBench examples: 187542
Loading tokenizer ...
config.json: 100% 661/661 [00:00<00:00, 2.96MB/s]
tokenizer_config.json: 7.30kB [00:00, 17.9MB/s]
vocab.json: 2.78MB [00:00, 67.6MB/s]
merges.txt: 1.67MB [00:00, 131MB/s]
tokenizer.json: 7.03MB [00:00, 173MB/s]
Parsing ToolBench examples (strict: tool_call+result+answer required) ...
Parsed  : 200 complete chain examples
Skipped : 806 (missing t

In [3]:
!python /content/sample_data/ADV_STEP2_multiturn_grpo.py

Loading ToolBench dataset ...
  Columns: ['id', 'conversations']
Extracting prompts ...
Loaded 50 prompts (skipped 0)

Loading tokenizer ...
Loading base model + SFT adapter ...
Loading weights: 100% 434/434 [00:23<00:00, 18.53it/s, Materializing param=model.norm.weight] 
Model ready.

  TRUE MULTI-TURN GRPO TRAINING
  Model          : Qwen2.5-3B
  Prompts        : 50
  Rollouts/step  : 2
  Max turns/chain: 4
  Train steps    : 40

  Key fixes active:
  • Wrap-up injected IMMEDIATELY when ≥2 tool calls made
  • Wrap-up injected on repeated identical tool call
  • Repeated calls penalised in reward (-0.3 each)
  • Non-empty args rewarded (+0.15 each)

Step  10/40 | loss=0.3938 | avg_reward=0.818 | chain=[TOOL → TOOL → TOOL → TOOL] reward=0.70
Step  20/40 | loss=0.6510 | avg_reward=0.505 | chain=[TOOL → TOOL → ANSWER] reward=1.00
Step  30/40 | loss=2.9954 | avg_reward=0.540 | chain=[TOOL → TOOL → ANSWER] reward=0.62
Step  40/40 | loss=0.3568 | avg_reward=0.715 | chain=[TOOL → TOOL → ANSW

In [4]:
!python /content/sample_data/ADV_STEP3_multiturn_eval.py


Loading SFT model ...
Loading weights: 100% 434/434 [00:20<00:00, 21.13it/s, Materializing param=model.norm.weight] 
  SFT model ready.

Loading GRPO model ...
Loading weights: 100% 434/434 [00:05<00:00, 78.23it/s, Materializing param=model.norm.weight] 
  GRPO model ready.

              MULTI-TURN TOOL CALL EVALUATION: SFT vs GRPO              

[01] Weather + Packing Advice
  User    : I'm travelling to Tokyo next week. What's the weather like and what should I pac
  Expected: get_weather → final_answer

  SFT  [0.90]
    Turn 1 [TOOL]  : weather_forecast({"location": "Tokyo"})
    Turn 2 [ANSWER]: Based on the weather forecast for Tokyo, it will be rainy all week with a temperature of 1
    Args populated : 1/1 tool calls
    Repeated calls : 0
    Got answer     : True

  GRPO [0.90]
    Turn 1 [TOOL]  : weather_forecast({"city": "Tokyo"})
    Turn 2 [ANSWER]: The weather in Tokyo will be rainy all week with a temperature of 12 degrees Celsius and h
    Args populated : 1/1 tool 